In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

### Setup
- ready to run for a prefect run with end_WY=2025 and start_WY=2024

In [ ]:
spark = create_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# define POR for recent test
wy = 2025
end_date = pd.to_datetime(f'09-30-{wy}')

### Check primary

In [ ]:
obs_df = ev.primary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()

obs_df.head()

In [ ]:
print(obs_df.shape)

In [ ]:
var_list = obs_df['variable_name'].unique()
print(var_list)

### Check Secondary

In [ ]:
fcst_df = ev.secondary_timeseries.filter(
    f"value_time < '{end_date}'"
).to_pandas()
fcst_df.head()

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

### Check single site

#### Fcst

In [ ]:
fcst_df = ev.secondary_timeseries.filter([
    "location_id = 'cbrfc-OAWU1'",
    f"value_time < '{end_date}'",
    "member = 'crx'",
]).order_by(
    'value_time'
).to_pandas()

fcst_df.tail(15)

In [ ]:
print(fcst_df.shape)

In [ ]:
var_list = fcst_df['variable_name'].unique()
print(var_list)

#### Obs

In [ ]:
obs_df = ev.primary_timeseries.filter([
    "location_id = 'cbrfc-OAWU1'",
    f"value_time < '{end_date}'"
]).order_by(
    'value_time'
).to_pandas()

obs_df.head(15)

In [ ]:
obs_df.tail(30)

In [ ]:
print(obs_df.shape)

In [ ]:
obs_var_list = obs_df['variable_name'].unique()
print(obs_var_list)

### Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from typing import Tuple, List


def plot_seasonal_forecast_comparison_multi_wy(
    location_id: str,
    primary_timeseries: pd.DataFrame,
    secondary_timeseries: pd.DataFrame,
    water_years: List[int],
) -> Tuple[plt.Figure, List[plt.Axes]]:
    """
    Plot observed and forecast water supply data for a location across multiple water years.
    
    Shows for each WY:
    - Observed cumulative sum (primary_timeseries, volume_*_cumsum) by value_time
    - Final observed value from final observed timeseries (primary_timeseries, volume_*_sum) by reference_time
    - CRX ensemble forecast (secondary_timeseries, highlighted in bold) by reference_time
    - Other ensemble members (secondary_timeseries, faded background) by reference_time
    
    Parameters
    ----------
    location_id : str
        The location ID to plot (e.g., 'cbrfc-american')
    primary_timeseries : pd.DataFrame
        Primary timeseries table containing observed data
    secondary_timeseries : pd.DataFrame
        Secondary timeseries table containing forecast data
    water_years : List[int]
        List of water years to plot (e.g., [2023, 2024, 2025])
        
    Returns
    -------
    fig, axes : matplotlib Figure and list of Axes objects
    """
    
    def get_wy_range(wy: int) -> Tuple[pd.Timestamp, pd.Timestamp]:
        """Get Oct 1 (start) and Sept 30 (end) for a given water year."""
        start = pd.Timestamp(year=wy - 1, month=10, day=1)
        end = pd.Timestamp(year=wy, month=9, day=30)
        return start, end
    
    def filter_by_wy(df: pd.DataFrame, wy: int, time_column: str = 'value_time') -> pd.DataFrame:
        """Filter dataframe to only include data within a water year."""
        start, end = get_wy_range(wy)
        return df[(df[time_column] >= start) & (df[time_column] <= end)].copy()
    
    # Filter data for location
    obs_data = primary_timeseries[primary_timeseries['location_id'] == location_id].copy()
    fcst_data = secondary_timeseries[secondary_timeseries['location_id'] == location_id].copy()
    
    if obs_data.empty and fcst_data.empty:
        raise ValueError(f"No data found for location_id: {location_id}")
    
    # Ensure datetime columns
    obs_data['value_time'] = pd.to_datetime(obs_data['value_time'])
    obs_data['reference_time'] = pd.to_datetime(obs_data['reference_time'])
    fcst_data['value_time'] = pd.to_datetime(fcst_data['value_time'])
    fcst_data['reference_time'] = pd.to_datetime(fcst_data['reference_time'])
    
    # Determine forecast period (extract from variable name: volume_MMDD_sum)
    if not fcst_data.empty:
        sample_var = fcst_data['variable_name'].iloc[0]
        forecast_period = sample_var.split('_')[1]  # e.g., '0407' from 'volume_0407_sum'
    else:
        forecast_period = "XXXX"
    
    # Create subplots
    n_wys = len(water_years)
    fig, axes = plt.subplots(1, n_wys, figsize=(6 * n_wys, 7))
    
    # Handle single subplot case (axes will be a single Axes object, not an array)
    if n_wys == 1:
        axes = [axes]
    
    # Plot each water year
    for idx, wy in enumerate(water_years):
        ax = axes[idx]
        
        # Filter data for this water year
        # For cumsum observations, filter by value_time
        obs_cumsum_wy = filter_by_wy(obs_data, wy, time_column='value_time')
        obs_cumsum_wy = obs_cumsum_wy[obs_cumsum_wy['variable_name'] == f'volume_{forecast_period}_cumsum'].copy()
        
        # For final observed, filter by value_time (they align with forecast period start)
        obs_final_wy = filter_by_wy(obs_data, wy, time_column='value_time')
        obs_final_wy = obs_final_wy[obs_final_wy['variable_name'] == f'volume_{forecast_period}_sum'].copy()
        
        # For forecasts, filter by reference_time (when forecast was issued)
        fcst_wy = filter_by_wy(fcst_data, wy, time_column='reference_time')
        
        # Sort by appropriate time column
        if not obs_cumsum_wy.empty:
            obs_cumsum_wy = obs_cumsum_wy.sort_values('value_time')
        if not obs_final_wy.empty:
            obs_final_wy = obs_final_wy.sort_values('reference_time')
        
        # Plot ensemble forecast members by reference_time
        if not fcst_wy.empty:
            members = fcst_wy['member'].unique()
            
            # Plot non-CRX members in background (light gray, thin)
            for member in members:
                if member.lower() != 'crx':
                    member_data = fcst_wy[fcst_wy['member'] == member].sort_values('reference_time')
                    ax.plot(
                        member_data['reference_time'],
                        member_data['value'],
                        color='gray',
                        alpha=0.25,
                        linewidth=1,
                        label='Other members' if member == members[members != 'crx'][0] else '',
                        zorder=1
                    )
            
            # Plot CRX in foreground (bold)
            crx_data = fcst_wy[fcst_wy['member'].str.lower() == 'crx'].sort_values('reference_time')
            if not crx_data.empty:
                ax.plot(
                    crx_data['reference_time'],
                    crx_data['value'],
                    color='#1f77b4',
                    linewidth=3,
                    label='Forecast (CRX)',
                    zorder=3,
                    marker='o',
                    markersize=4
                )
        
        # Plot observed cumsum by value_time
        if not obs_cumsum_wy.empty:
            ax.plot(
                obs_cumsum_wy['value_time'],
                obs_cumsum_wy['value'],
                color='#2ca02c',
                linewidth=2.5,
                label='Observed (cumulative)',
                zorder=2,
                marker='s',
                markersize=4
            )
        
        # Plot final observed value by reference_time (single point)
        if not obs_final_wy.empty:
            # Plot point at reference_time
            ax.scatter(
                obs_final_wy['reference_time'],
                obs_final_wy['value'],
                color='#d62728',
                s=200,
                marker='*',
                label='Observed (final)',
                zorder=4,
                edgecolors='darkred',
                linewidth=1.5
            )
            
            # Add horizontal reference line from final value
            final_value = obs_final_wy.iloc[0]['value']
            ax.axhline(
                y=final_value,
                color='#d62728',
                linestyle='--',
                alpha=0.3,
                linewidth=1,
                zorder=0
            )
        
        # Formatting
        ax.set_xlabel('Date', fontsize=11, fontweight='bold')
        if idx == 0:  # Only leftmost plot has y-label
            ax.set_ylabel('Water Supply Volume (Acre-Feet)', fontsize=11, fontweight='bold')
        
        ax.set_title(f'WY {wy}', fontsize=12, fontweight='bold')
        
        # Format x-axis dates
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
        
        # Grid and legend
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(loc='best', fontsize=10, framealpha=0.95)
        
        # Add minor gridlines
        ax.minorticks_on()
        ax.grid(True, which='minor', alpha=0.1, linestyle=':')
    
    # Overall title
    fig.suptitle(
        f'Seasonal Water Supply Forecast vs. Observed - {location_id} (Forecast Period: {forecast_period})',
        fontsize=14,
        fontweight='bold',
        y=1.00
    )
    
    fig.tight_layout()
    
    return fig, axes

In [ ]:
fig, ax = plot_seasonal_forecast_comparison_multi_wy(
    location_id='cbrfc-OAWU1',
    primary_timeseries=obs_df,
    secondary_timeseries=fcst_df,
    water_years=[wy-1,wy]
)
plt.show()

### Kill spark

In [ ]:
ev.spark.stop()